# `runner.ipynb` — universal stage runner

**What this does**
1. Mounts Google Drive (Colab only) so the cache + outputs persist across sessions. Locally, everything just lives on disk in this repo checkout.
2. Sources every code cell from `basic_cells.ipynb` (no copy-paste — single source of truth).
3. Reads one config YAML and calls `run_stage(...)`.
4. Saves outputs under `<output_dir>/<stage_id>/` (config snapshot, predictions, metrics, plots, summary).

**To run on Colab**
- Upload this repo's `path_a/` contents + `algo_data/data/processed/...` to `MyDrive/moex-hack/` (see `path_a/scratchpads/phase_b_scratch_pad.md` for the exact layout that's been tested).
- Edit the `CONFIG_PATH` cell below.
- Runtime → Run all.

**To run locally (no Colab, no Drive)**
- Open this notebook from within a Jupyter/VS Code kernel — §0 auto-detects it's not Colab and resolves `PROJECT_DIR` to wherever `basic_cells.ipynb` actually is (works whether the kernel's cwd is `path_a/` or the repo root).
- Needs `chronos-forecasting`, `pandas[pyarrow]`, `requests`, `matplotlib`, `numpy`, `tqdm`, `pyyaml`, `scipy`, `nbformat` installed in the local Python environment (same list §1's Colab `!pip install` cell uses — install them yourself first, that cell is Colab-only).
- No GPU needed for this to *work* — `basic_cells.ipynb` §1 already falls back to `DEVICE="cpu"`/`DTYPE=torch.float32` automatically — but every `predict_df` call is a real forward pass through the model, and Phase B's configs run hundreds of them (`max_windows: 400` × 2 arms), so expect this to take meaningfully longer than on a GPU. Time a handful of windows first if you want a realistic ETA before committing to a full run.
- `algopack_processed_path` in the Phase B configs resolves automatically for both layouts (flat Colab-Drive layout, or this repo's `path_a/` + `../algo_data/` local layout) — see `load_config` in `basic_cells.ipynb` §2.

**To run in parallel**
- Open a second Colab session, same notebook file, point `CONFIG_PATH` at a different config. Cache is shared (Drive); outputs are namespaced by `stage_id`.

**First time only**
- Run the optional "bulk prefetch" cell once to fill the cache — softer on ISS than letting every run prefetch its own slice on first run.

**Note**: the old numbered-stage configs (stage_0 through stage_7) are archived at `path_a/archive/concluded_stages/` — the project pivoted from a linear stage sequence to gated phases (A–D), see `docs/exp_plan.md`.

## 0. Locate the project

On Colab: set `PROJECT_DIR` to the folder containing this notebook + `basic_cells.ipynb` + `configs/` (default assumes `MyDrive/moex-hack/`). Locally: auto-detected below — resolves to wherever `basic_cells.ipynb` actually sits, so it works whether you launched the kernel from `path_a/` or the repo root. If it still gets it wrong (unusual folder layout), set `PROJECT_DIR` by hand before running the next cell.

In [ ]:
import os
from pathlib import Path

IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in str(type(globals().get("get_ipython", lambda: None)()))
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive", force_remount=False)
    PROJECT_DIR = "/content/drive/MyDrive/moex-hack"
else:
    # Local run: PROJECT_DIR must be the folder containing basic_cells.ipynb + configs/
    # (i.e. path_a/ in this repo). Path.cwd() only gives the right answer if the
    # kernel was launched with path_a/ as the working directory -- if this notebook
    # was opened from elsewhere (repo root, a different folder), fall back to
    # resolving relative to this .ipynb file's own location instead of guessing.
    candidates = [Path.cwd(), Path.cwd() / "path_a"]
    PROJECT_DIR = next((str(c) for c in candidates if (c / "basic_cells.ipynb").exists()), str(Path.cwd()))

os.chdir(PROJECT_DIR)
print(f"PROJECT_DIR = {PROJECT_DIR}")
print("contents:", sorted(os.listdir(PROJECT_DIR)))
if not IN_COLAB and not (Path(PROJECT_DIR) / "basic_cells.ipynb").exists():
    print("WARNING: basic_cells.ipynb not found here -- set PROJECT_DIR manually to "
          "the folder that contains it (locally: the repo's path_a/ folder) before "
          "continuing to \xa71.")


## 1. Source `basic_cells.ipynb`

Executes every code cell from the cell library inside this kernel — no imports, no module packaging, no version drift.

In [ ]:
import json, nbformat
from IPython import get_ipython

BASIC = Path(PROJECT_DIR) / "basic_cells.ipynb"
assert BASIC.exists(), f"missing {BASIC} — copy it next to this notebook"
nb = nbformat.read(str(BASIC), as_version=4)
ip = get_ipython()
n_code = 0
for cell in nb.cells:
    if cell.cell_type == "code":
        ip.run_cell(cell.source)
        n_code += 1
print(f"sourced {n_code} code cells from basic_cells.ipynb")


## 2. (Optional, one-time) Bulk prefetch all stages

Run once on a fresh Drive cache. Idempotent — safe to re-run.

In [ ]:
# Uncomment to bulk-prefetch every configured run's data into the cache.
# Glob covers both the retired stage_*.yaml naming (archive/concluded_stages/, if copied back) and the current phase_*.yaml naming.
all_cfgs = [load_config(str(p)) for p in sorted(Path("configs").glob("phase_*.yaml"))]
cache_dir = resolve_cache_dir(all_cfgs[0])
manifest = build_prefetch_manifest(all_cfgs)
print(f"manifest size: {len(manifest)} files")
prefetch_all(manifest, cache_dir)

## 3. Pick a config and run

Edit `CONFIG_PATH` to point at the config you want to run.

In [ ]:
CONFIG_PATH = "configs/phase_b_multivariate.yaml"  # <-- edit me: phase_b_multivariate.yaml / phase_b_univariate.yaml
# Phase B gate: ran 2026-09-17, GATE FAILED (both arms chance-level, 0/64 BH-significant cells,
# aggregate ΔDA ≈ 0). See path_a/scratchpads/phase_b_scratch_pad.md for the full result and
# docs/current_state.md session entry 17. Phase C/D not funded per the pre-registered rule.

summary = run_stage(CONFIG_PATH)


## 4. Inspect outputs in-line

After `run_stage` finishes, the cell below loads the metrics tables for quick review without leaving the notebook.

In [ ]:
cfg = load_config(CONFIG_PATH)
stage_out = Path(cfg["output_dir"]) / cfg["stage_id"]
print("outputs at:", stage_out)
print("files:", sorted(p.name for p in stage_out.rglob("*") if p.is_file()))

import pandas as pd
metrics = pd.read_csv(stage_out / "metrics.csv")
agg     = pd.read_csv(stage_out / "metrics_aggregate.csv")
print("\n--- aggregate ---");        print(agg.to_string(index=False))
print("\n--- per-cell (head) ---");  print(metrics.head(20).to_string(index=False))

# Display every plot saved by run_stage
from IPython.display import Image, display
for p in sorted((stage_out / "plots").glob("*.png")):
    print(p.name); display(Image(str(p)))


## 5. Notes

- **Cache only**: `run_stage` reads from Parquet cache. If a file is missing it errors loudly with the exact missing key — fix the config / re-run §2 prefetch.
- **Multiple runs in one session**: just re-edit `CONFIG_PATH` and re-run §3 + §4. Outputs are namespaced by `stage_id`; nothing is overwritten across runs.
- **Long runs**: `run_walk_forward` checkpoints `preds_partial.parquet` every 25 windows.
- **Path B**: when fine-tuning is wired in, it lives behind a `cfg["path_b"]["enabled"]` flag — same runner.